# UniSkill methodology demonstration with Logistic Regression
Research context: Comparing Machine Learning Methods for Matching Mentors’ Teaching Competencies
with Faculty Members’ Professional Development Needs in Higher Education.

This notebook tests a **competency–course matching subsystem**, not actual mentor assignment.
It uses only author-annotated real UniSkill records. No synthetic rows, new labels, or generated course texts are used.
Authors' predictions are excluded from inputs. The combined target is `eval_all`.

Source: Musazade, Mezei, and Zhang (2026), https://aclanthology.org/2026.lrec-1.31/.
Data: https://huggingface.co/datasets/nurlanm/UniSkill_dataset.
The training source is filtered by `data_source == "real"`; validation and evaluation files contain real annotations
according to paper section 3.3.1. All are pooled and independently split for this demonstration.
Results are not a reproduction of the authors' benchmark.
Dataset card license: Apache-2.0.

Unzip the evidence bundle, install requirements, open this notebook, and select **Restart Kernel and Run All**.
Run from the extracted folder. Every cell uses local data; no cloud API is required.

Assignment 3 demonstrates ONE model. The wider research title concerns comparison, but no model-comparison claim is made here.

In [ ]:
from pathlib import Path
import hashlib, json, platform, re, time, importlib.metadata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

BASE = Path.cwd()
RESULT = BASE / 'results'
RESULT.mkdir(exist_ok=True)
SEED = 42
versions = {'python': platform.python_version(), 'platform': platform.platform(),
            **{p: importlib.metadata.version(p) for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'openpyxl']}}
print(json.dumps(versions, indent=2))
manifest = json.loads((BASE / 'source_manifest.json').read_text())
assert hashlib.sha256((BASE / 'manual_records.csv').read_bytes()).hexdigest() == manifest['manual_csv_sha256']
raw = pd.read_csv(BASE / 'manual_records.csv')
print('Manual source rows:', len(raw))
print(raw.groupby('source_file').size().to_string())
print('Missing skill descriptions:', raw.skill_description.isna().sum())

{
  "python": "3.12.14",
  "platform": "macOS-27.0-arm64-arm-64bit",
  "pandas": "3.0.6",
  "numpy": "2.5.4",
  "scipy": "1.18.1",
  "scikit-learn": "1.9.1",
  "openpyxl": "3.1.5"
}
Manual source rows: 1098
source_file
evaldf_public_m.xlsx          301
traindf_public_m.xlsx         695
validationdf_public_m.xlsx    102
Missing skill descriptions: 6


## Clean without inventing data
Missing descriptions are replaced by the empty string; the original skill label remains.
Existing labels are retained. Exact input duplicates are removed only after confirming agreement.
The original spreadsheet row is retained for traceability.

In [ ]:
text_cols = ['skill_label', 'skill_description', 'matched_course_sentence', 'course_title']
df = raw.copy()
for col in text_cols:
    df[col] = df[col].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
assert not df[['skill_label', 'matched_course_sentence', 'course_title']].eq('').any().any()
assert set(df.eval_all.unique()) <= {0, 1}
assert (df.eval_all == (df.eval_all_tit * df.eval_all_sen)).all()
assert df.groupby(text_cols, dropna=False).eval_all.nunique().max() == 1
duplicate_count = int(df.duplicated(text_cols).sum())
df = df.drop_duplicates(text_cols).reset_index(drop=True)
df['record_id'] = [f'R{i:04d}' for i in range(len(df))]
df['skill_text'] = df.skill_label + ' ' + df.skill_description
print('Removed duplicate rows:', duplicate_count)
print('Clean rows:', len(df), 'Class counts:', df.eval_all.value_counts().sort_index().to_dict())

Removed duplicate rows: 2
Clean rows: 1096 Class counts: {0: 700, 1: 396}


## Connected groups and independent splits
Records sharing an identical normalized course title OR course sentence belong to the same connected group.
Groups cannot cross train, validation, and test. The 60/20/20 targets apply to groups; row counts can differ.
Skills may recur across splits: this tests new course content for a known skill vocabulary, not unseen skills.

In [ ]:
parent = list(range(len(df)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
def union(i, j):
    parent[find(j)] = find(i)
for col in ['course_title', 'matched_course_sentence']:
    seen = {}
    for i, text in enumerate(df[col]):
        key = text.casefold()
        if key in seen:
            union(i, seen[key])
        else:
            seen[key] = i
df['group'] = [find(i) for i in range(len(df))]
first = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
dev_idx, test_idx = next(first.split(df, groups=df.group))
dev = df.iloc[dev_idx]
second = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_local, val_local = next(second.split(dev, groups=dev.group))
train_idx, val_idx = dev_idx[train_local], dev_idx[val_local]
parts = {'train': train_idx, 'validation': val_idx, 'test': test_idx}
for a, b in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert not (set(df.iloc[parts[a]].group) & set(df.iloc[parts[b]].group))
    for col in ['course_title', 'matched_course_sentence']:
        assert not (set(df.iloc[parts[a]][col].str.casefold()) & set(df.iloc[parts[b]][col].str.casefold()))
df['split'] = ''
for split, idx in parts.items():
    df.loc[idx, 'split'] = split
    assert df.iloc[idx].eval_all.nunique() == 2
print(pd.crosstab(df.split, df.eval_all).to_string())
print('Connected groups:', df.group.nunique(), 'Largest group:', df.groupby('group').size().max())
print('Course-title and course-sentence overlap across splits: 0')
df.to_csv(RESULT / 'clean_records_and_splits.csv', index=False)

eval_all      0    1
split               
test        132   84
train       394  255
validation  174   57
Connected groups: 544 Largest group: 21
Course-title and course-sentence overlap across splits: 0


## Fit TF-IDF on training text only
Five matching features: sentence-to-skill cosine, title-to-skill cosine, their minimum,
sentence-to-skill-label Jaccard overlap, and title-to-skill-label Jaccard overlap.
This small feature set is an initial lexical baseline. It does not measure pedagogical expertise.

In [ ]:
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1, max_features=20000, norm='l2')
train = df.iloc[train_idx]
vectorizer.fit(pd.concat([train.course_title, train.matched_course_sentence, train.skill_text]).tolist())
skills = vectorizer.transform(df.skill_text)
titles = vectorizer.transform(df.course_title)
sentences = vectorizer.transform(df.matched_course_sentence)
cos_s = np.asarray(sentences.multiply(skills).sum(axis=1)).ravel()
cos_t = np.asarray(titles.multiply(skills).sum(axis=1)).ravel()
def jaccard(a, b):
    a, b = set(re.findall(r'\b\w+\b', a.casefold())), set(re.findall(r'\b\w+\b', b.casefold()))
    return len(a & b) / len(a | b) if a | b else 0.0
jac_s = np.array([jaccard(a, b) for a, b in zip(df.matched_course_sentence, df.skill_label)])
jac_t = np.array([jaccard(a, b) for a, b in zip(df.course_title, df.skill_label)])
X = np.column_stack([cos_s, cos_t, np.minimum(cos_s, cos_t), jac_s, jac_t])
y = df.eval_all.astype(int).to_numpy()
assert np.isfinite(X).all()
print('Feature matrix:', X.shape, 'TF-IDF vocabulary:', len(vectorizer.vocabulary_))

Feature matrix: (1096, 5) TF-IDF vocabulary: 15313


## Validation selection and held-out evaluation
Model settings are fixed before evaluation. The Logistic Regression decision threshold is selected on validation macro-F1.
Only the selected threshold is applied to test. Only one ML model, Logistic Regression, is trained in this demonstration.
No published prediction columns or test labels enter training or threshold selection.

In [ ]:
models = {
    'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=1000, class_weight='balanced', random_state=SEED)),
}
thresholds = np.linspace(0.05, 0.95, 19)
results, predictions, errors, configs = [], {}, [], {}
probabilities = {}
train_times = {}
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X[train_idx], y[train_idx])
    train_times[name] = time.perf_counter() - start
    probabilities[name] = model.predict_proba(X)[:, 1]
    configs[name] = str(model)
threshold_log = []
for name, prob in probabilities.items():
    scores = [(f1_score(y[val_idx], prob[val_idx] >= t, average='macro', zero_division=0), t) for t in thresholds]
    best = max(s for s, t in scores)
    threshold = min((t for s, t in scores if s == best), key=lambda t: (abs(t - 0.5), t))
    predictions[name] = (prob[test_idx] >= threshold).astype(int)
    threshold_log.append({'method': name, 'threshold': float(threshold), 'validation_macro_f1': float(best)})
for name, pred in predictions.items():
    cm = confusion_matrix(y[test_idx], pred, labels=[0, 1])
    row = {'method': name, 'accuracy': float(accuracy_score(y[test_idx], pred)),
        'precision_match': float(precision_score(y[test_idx], pred, zero_division=0)),
        'recall_match': float(recall_score(y[test_idx], pred, zero_division=0)),
        'f1_match': float(f1_score(y[test_idx], pred, zero_division=0)),
        'macro_f1': float(f1_score(y[test_idx], pred, average='macro', zero_division=0)),
        'TN': int(cm[0,0]), 'FP': int(cm[0,1]), 'FN': int(cm[1,0]), 'TP': int(cm[1,1])}
    results.append(row)
metrics = pd.DataFrame(results)
print('Validation choices:')
print(pd.DataFrame(threshold_log).to_string(index=False))
print('Held-out test metrics:')
print(metrics.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
metrics.to_csv(RESULT / 'test_metrics.csv', index=False)
pd.DataFrame(threshold_log).to_csv(RESULT / 'validation_thresholds.csv', index=False)
test_output = df.iloc[test_idx].copy()
for name, pred in predictions.items():
    test_output[name] = pred
test_output.to_csv(RESULT / 'test_predictions.csv', index=False)
report = {'seed': SEED, 'manual_raw_rows': len(raw), 'synthetic_training_rows_excluded': 3212,
    'clean_rows': len(df), 'removed_duplicates': duplicate_count,
    'missing_descriptions': int(raw.skill_description.isna().sum()),
    'classes': {str(k): int(v) for k,v in df.eval_all.value_counts().sort_index().items()},
    'groups': int(df.group.nunique()), 'split_sizes': {s: len(idx) for s,idx in parts.items()},
    'split_class_counts': {s: {str(k): int(v) for k,v in df.iloc[idx].eval_all.value_counts().sort_index().items()} for s,idx in parts.items()},
    'metrics': results, 'validation_choices': threshold_log, 'versions': versions,
    'model_configs': configs, 'train_seconds': train_times,
    'limitation': 'Lexical feature demonstration on real annotated course-skill pairs; no real mentor matching or pedagogical outcome validation.'}
(RESULT / 'run_summary.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

Validation choices:
             method  threshold  validation_macro_f1
Logistic regression       0.55              0.73687
Held-out test metrics:
             method  accuracy  precision_match  recall_match  f1_match  macro_f1  TN  FP  FN  TP
Logistic regression    0.7269           0.7049        0.5119    0.5931    0.6938 114  18  41  43


1725

## Inspect errors rather than inventing examples
The following records are actual held-out observations. A disagreement with the reference label is a model error
for this benchmark; the annotation itself may also be uncertain. Macro-F1 reflects both classes.

In [ ]:
focus = 'Logistic regression'
error_df = test_output.loc[test_output[focus] != test_output.eval_all,
    ['record_id', 'skill_label', 'course_title', 'matched_course_sentence', 'eval_all', focus, 'source_file', 'source_excel_row']]
print('Logistic regression errors:', len(error_df))
print(error_df.head(3).to_string(index=False))
error_df.to_csv(RESULT / 'logistic_regression_errors.csv', index=False)

Logistic regression errors: 59
record_id                                     skill_label                                                               course_title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     matched_course_sentence  eval_all  Logistic regression           source_file  source_excel_row
    R0007                             SAS Data Management                                                   Data Modeling and Des

## Interpretation and limits
This is an actual Logistic Regression run on a small, grouped split. It demonstrates initial pipeline readiness.
The original authors' split is not reused and their reported performance is not our result.
Author predictions were excluded. Missing descriptions are not synthesized.
A small lexical feature set can confuse word overlap with meaning and miss implicit competencies.
The benchmark concerns course relevance, not mentors' verified competencies or faculty development needs.
Comparing alternative models remains part of the wider research project, beyond this assignment demonstration.

References: UniSkill https://doi.org/10.63317/2n39qzvk2eqe;
scikit-learn https://jmlr.org/papers/v12/pedregosa11a.html;
https://scikit-learn.org/stable/modules/feature_extraction.html;
https://scikit-learn.org/stable/modules/model_evaluation.html.